# Proyecto 2: Competencia de Agentes en Space Invaders

CC3092 Deep Learning y Sistemas Inteligentes

Este notebook documenta el análisis del entorno, la metodología y los resultados del agente DQN entrenado para `ALE/SpaceInvaders-v5`. Construido sobre las utilidades del Laboratorio #5 (`ale_utils.py`) y el paquete `dqn/` de este repositorio.

Repositorio: https://github.com/AngelEsquit/PRY2-DL

In [ ]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ale_utils import crear_entorno, agente_aleatorio, agente_regla_simple, ejecutar_episodio
from dqn.wrappers import crear_entorno_dqn, ENV_ID, FRAME_SIZE, N_FRAME_STACK, FRAME_SKIP


## 1. Definición del problema y análisis del entorno

### 1.1 Descripción del entorno

`ALE/SpaceInvaders-v5` simula el juego de Atari 2600 *Space Invaders*: el jugador controla una nave que se mueve horizontalmente en la parte inferior de la pantalla y dispara contra hileras de invasores alienígenas que descienden progresivamente. El objetivo es destruir la mayor cantidad de invasores posible antes de que estos lleguen a la base del jugador o de perder las 3 vidas disponibles.

- **Recompensa**: +puntos al destruir un invasor (varía según la fila/tipo de invasor destruido; los invasores de filas superiores valen más puntos), sin penalización explícita por perder o disparar.
- **Fin de episodio**: `terminated=True` cuando el jugador pierde sus 3 vidas (game over) o los invasores llegan a la base; `truncated=True` si se alcanza el límite de pasos del entorno (`TimeLimit`, por defecto grande para Atari).
- **Vidas**: 3 vidas por partida (consultable vía `info["lives"]` / `env.unwrapped.ale.lives()`).

In [ ]:
env = crear_entorno(ENV_ID)
obs, info = env.reset(seed=0)

print("Espacio de observación:", env.observation_space)
print("Espacio de acción (Discrete):", env.action_space)
print("Significado de cada acción:", env.unwrapped.get_action_meanings())
print("Vidas iniciales:", info.get("lives"))
env.close()

### 1.2 Espacio de observación y de acción

- **Observación cruda**: imagen RGB de `(210, 160, 3)` píxeles por defecto (`obs_type="rgb"`). ALE también permite `obs_type="grayscale"` o `obs_type="ram"` (128 bytes de la RAM del emulador).
- **Espacio de acción mínimo** (`full_action_space=False`, por defecto): `Discrete(6)` — `NOOP, FIRE, RIGHT, LEFT, RIGHTFIRE, LEFTFIRE`. El espacio de acción completo de ALE (`full_action_space=True`) tiene 18 acciones, muchas de las cuales no tienen efecto en este juego (p. ej. `UP`, `DOWN`).

**Decisión de diseño**: se usa el subconjunto mínimo de 6 acciones (`full_action_space=False`). Ampliar a las 18 acciones no agrega expresividad útil para este juego (el jugador solo se mueve horizontalmente) y sí incrementa el espacio de exploración que la red debe aprender a descartar, ralentizando el entrenamiento.

### 1.3 Análisis de la señal de recompensa

Para caracterizar qué tan densa/dispersa es la recompensa, ejecutamos un episodio con el agente aleatorio del Laboratorio #5 y observamos en qué fracción de los pasos se recibe recompensa distinta de cero.

In [ ]:
env = crear_entorno(ENV_ID)
obs, info = env.reset(seed=0)

rewards = []
terminated = truncated = False
while not (terminated or truncated):
    action = agente_aleatorio(obs, env)
    obs, reward, terminated, truncated, info = env.step(action)
    rewards.append(reward)
env.close()

rewards = np.array(rewards)
print(f"Pasos totales: {len(rewards)}")
print(f"Pasos con recompensa > 0: {(rewards > 0).sum()} ({100*(rewards>0).mean():.2f}%)")
print(f"Recompensa total del episodio: {rewards.sum():.1f}")
print(f"Valores de recompensa observados: {sorted(set(rewards.tolist()))}")

**Interpretación** (completar/ajustar tras ejecutar con la semilla final): la recompensa es **dispersa** — la gran mayoría de los pasos (moverse, disparar sin acertar) reciben recompensa 0, y solo una fracción pequeña de pasos (impactar un invasor) produce recompensa positiva. Además, la magnitud de la recompensa varía según qué invasor se destruye (5 a 30 puntos aprox. según la fila), lo que en un problema con muchos juegos de Atari de escalas distintas motiva **recortar (clip) la recompensa a {-1, 0, 1}** durante el entrenamiento (como en Mnih et al., 2015), para que la magnitud del error de TD sea comparable entre pasos y estabilizar el entrenamiento. En evaluación/competencia se reporta la recompensa real sin recortar, como exige el enunciado.

No se observó necesidad de *reward shaping* adicional (recompensas artificiales intermedias): la señal de recompensa original, aunque dispersa, es suficientemente frecuente en Space Invaders (hay muchos invasores que destruir) para que el replay buffer acumule transiciones informativas sin ayuda externa.

### 1.4 Preprocesamiento y decisiones de diseño

Implementado en `dqn/wrappers.py::crear_entorno_dqn`, construido *encima* de `ale_utils.crear_entorno` (no lo reemplaza):

| Paso | Wrapper | Motivo |
|---|---|---|
| Escala de grises + resize 84x84 | `AtariPreprocessing` | El color no es relevante para identificar naves/invasores/proyectiles; reduce drásticamente el tamaño de entrada de la CNN. |
| Frame-skip = 4 con max-pool de 2 frames | `AtariPreprocessing` | Los frames consecutivos son casi idénticos (poca información nueva) y algunos sprites parpadean cada frame en el hardware original de Atari; el max-pool evita perderlos. Repetir la acción 4 pasos también acelera el entrenamiento (menos decisiones de red por segundo de juego). |
| Apilado de 4 frames | `FrameStackObservation` | Una sola imagen no revela dirección/velocidad de las balas ni de los invasores (el entorno es parcialmente observable con un solo frame); apilar 4 frames se lo da al agente sin necesidad de una red recurrente. |
| `NOOP` aleatorio al inicio (hasta 30) | `AtariPreprocessing(noop_max=30)` | Evita que el agente memorice la secuencia exacta de frames iniciales de cada episodio. |
| Fin de episodio por vida perdida | `EpisodicLifeWrapper` (propio) | Solo en entrenamiento: al perder una vida se reporta `terminated=True` para propagar antes la señal negativa, sin reiniciar la partida real. |
| Reward clipping a {-1,0,1} | `ClipReward` | Solo en entrenamiento (ver sección 1.3). |

A continuación se visualiza un frame preprocesado (84x84, escala de grises) junto al frame RGB original.

In [ ]:
env_rgb = crear_entorno(ENV_ID, render_mode="rgb_array")
obs_rgb, _ = env_rgb.reset(seed=0)
frame_rgb = env_rgb.render()
env_rgb.close()

env_dqn = crear_entorno_dqn(terminal_on_life_loss=False, clip_reward=False)
obs_stack, _ = env_dqn.reset(seed=0)
env_dqn.close()

obs_stack = np.asarray(obs_stack)
print("Observación apilada:", obs_stack.shape, obs_stack.dtype)

fig, axes = plt.subplots(1, N_FRAME_STACK + 1, figsize=(14, 3))
axes[0].imshow(frame_rgb)
axes[0].set_title("RGB original (210x160)")
axes[0].axis("off")
for i in range(N_FRAME_STACK):
    axes[i + 1].imshow(obs_stack[i], cmap="gray")
    axes[i + 1].set_title(f"Frame apilado {i}")
    axes[i + 1].axis("off")
plt.tight_layout()
plt.show()

## 2. Metodología de desarrollo

### 2.1 Algoritmos considerados

Se implementó **DQN** (Mnih et al., 2015) desde cero en PyTorch (`dqn/agent.py`), con soporte para dos extensiones activables por configuración:

- **Double DQN** (Van Hasselt et al., 2016): la red *online* elige la mejor acción del siguiente estado y la red *objetivo* solo la evalúa, reduciendo la sobreestimación sistemática de los valores Q que sufre el DQN original (que usa la misma red objetivo para elegir y evaluar).
- **Dueling DQN** (Wang et al., 2016): arquitectura alternativa que separa la estimación de $V(s)$ y $A(s,a)$, combinadas como $Q(s,a) = V(s) + (A(s,a) - \text{mean}_{a'} A(s,a'))$.

Se descartó implementar PPO/A2C (métodos on-policy) porque DQN y sus variantes son más simples de depurar/comparar entre sí manteniendo la misma arquitectura convolucional y el mismo replay buffer, lo que facilita aislar el efecto de cada cambio (arquitectura vs. algoritmo de target) en la sección de resultados.

### 2.2 Arquitectura de la red

`NatureCNN` (`dqn/model.py`), la arquitectura convolucional original de Mnih et al. (2015):

| Capa | Detalle | Salida |
|---|---|---|
| Entrada | 4 frames apilados, 84x84, uint8 normalizado a [0,1] | (4, 84, 84) |
| Conv1 | 32 filtros, kernel 8x8, stride 4, ReLU | (32, 20, 20) |
| Conv2 | 64 filtros, kernel 4x4, stride 2, ReLU | (64, 9, 9) |
| Conv3 | 64 filtros, kernel 3x3, stride 1, ReLU | (64, 7, 7) |
| Flatten + FC | 512 unidades, ReLU | (512,) |
| Salida | `n_actions` valores Q (o V(s)+A(s,a) en Dueling) | (6,) |

### 2.3 Exploración vs. explotación

Epsilon-greedy con decaimiento **lineal** de $\epsilon=1.0$ a $\epsilon=0.01$ a lo largo de `epsilon_decay_steps` pasos de entorno (por defecto 1,000,000). Durante evaluación se usa política puramente greedy ($\epsilon=0$), según exige el protocolo de competencia.

### 2.4 Hiperparámetros de entrenamiento (configuración base)

| Hiperparámetro | Valor por defecto | Notas |
|---|---|---|
| Función de pérdida | Huber / Smooth L1 | Menos sensible a outliers en el error de TD que MSE |
| Optimizador | Adam | lr = 2.5e-4 |
| Factor de descuento γ | 0.99 | |
| Tamaño del replay buffer | 200,000 transiciones | Limitado por RAM disponible (cada transición: 2 × 4×84×84 uint8) |
| Tamaño de batch | 32 | |
| Pasos de calentamiento (`learning_starts`) | 50,000 | Pasos aleatorios antes de empezar a entrenar, para llenar el buffer |
| Frecuencia de entrenamiento | cada 4 pasos de entorno | |
| Frecuencia de actualización de la red objetivo | cada 10,000 pasos de gradiente | |

Estos valores son el punto de partida (iter01); la sección de resultados documenta qué cambios se probaron en iteraciones posteriores.

## 3. Entrenamiento

El entrenamiento completo (millones de pasos de entorno) se ejecuta desde línea de comandos, no desde el notebook, porque toma varias horas incluso con GPU:

```bash
python -m dqn.train --iteration iter01 --total-steps 2000000 --architecture dqn --double-dqn
```

Cada corrida registra una fila por episodio en `logs/<iteration>.csv` (columnas: `episode, env_step, reward_total, length, epsilon, avg_loss, elapsed_s`) y guarda el checkpoint en `checkpoints/<iteration>.pt`.

A continuación, una corrida muy corta (unos pocos miles de pasos) solo para verificar que el pipeline completo funciona end-to-end antes de lanzar el entrenamiento largo real.

In [ ]:
# Corrida de verificación (NO es entrenamiento real, solo confirma que el
# pipeline corre sin errores). Las iteraciones reales para el trabajo escrito
# se ejecutan por CLI con --total-steps del orden de 10^6-10^7.
!cd .. && python -m dqn.train --iteration demo_pipeline --total-steps 2000 --learning-starts 200 --buffer-size 5000 --target-update-freq 100 --checkpoint-freq 2000

## 4. Resultados de iteraciones

Se carga el CSV de cada iteración registrada en `logs/` y se grafica la recompensa de entrenamiento por episodio (promedio móvil) para comparar iteraciones. **Reemplazar la lista `iteraciones` por los identificadores reales usados al entrenar** (p. ej. `iter01`, `iter02`, ...).

In [ ]:
iteraciones = ["best_run"]  # ids reales de corridas de entrenamiento (ver checkpoints/, logs/)

plt.figure(figsize=(9, 5))
for it in iteraciones:
    df = pd.read_csv(f"../logs/{it}.csv")
    ventana = max(1, min(20, len(df)))
    df["reward_ma"] = df["reward_total"].rolling(ventana, min_periods=1).mean()
    plt.plot(df["env_step"], df["reward_ma"], label=it)

plt.xlabel("Pasos de entorno")
plt.ylabel("Recompensa de entrenamiento (promedio móvil)")
plt.title("Curvas de entrenamiento por iteración")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### Tabla resumen de iteraciones

**TODO**: seguir completando esta tabla a medida que se ejecuten iteraciones adicionales (ver `Recomendaciones` del enunciado: llevar este registro *durante* el entrenamiento, no reconstruirlo al final).

| Iteración | Cambios vs. anterior | Reward promedio (train, últimos ~150 ep.) | Reward promedio (evaluación greedy, 5 ep.) | Reward máximo (evaluación) | Observaciones |
|---|---|---|---|---|---|
| baseline aleatorio | — (sin aprendizaje) | — | 185.0 | 260.0 | `evaluate_baselines.py`; referencia de puntaje sin política |
| baseline regla simple | — (sin aprendizaje) | — | 270.0 | 270.0 | `evaluate_baselines.py`; política fija (disparo constante), determinística |
| best_run | Configuración base: Dueling DQN + Double DQN, γ=0.99, lr=2.5e-4, buffer=150k, ε: 1.0→0.02 en 1M pasos, 3,000,000 pasos totales | ~8-10 (recompensa recortada a {-1,0,1} durante entrenamiento, no comparable directamente con la recompensa real) | **710.0** | **800.0** | Episodios: [605, 715, 800, 715, 715]. ~3x el baseline de regla simple. Pérdida (`avg_loss`) estable y baja (~0.01) al final, sin señales de divergencia; posible margen de mejora con más pasos de entrenamiento |
| iter02 | ... | | | | |

**Problemas encontrados durante el entrenamiento**: ninguna divergencia de los valores Q observada (la pérdida se mantuvo estable y baja, ~0.01-0.02, durante todo el entrenamiento). No se observó estancamiento claro en una política subóptima dentro de los 3M pasos, aunque tampoco hay evidencia de que la curva haya convergido del todo — una iteración con más pasos de entrenamiento (p. ej. 6-8M) es la mejora más directa a probar a continuación.

## 5. Evaluación final y video

Una vez elegida la mejor iteración, se evalúa con política greedy (protocolo de competencia: 5 episodios, se reporta el máximo) y se genera el video final:

```bash
python ../evaluate.py --checkpoint ../checkpoints/<mejor_iteracion>.pt --episodes 5 --video-folder ../videos
```

El video generado (`videos/rl-video-episode-*.mp4`) es el mismo que se entrega como respaldo del resultado de competencia.

## 6. Discusión de resultados

**TODO** (completar con base en las iteraciones realmente ejecutadas):

- Comparación entre iteraciones: qué cambio (algoritmo, arquitectura, hiperparámetros) tuvo mayor impacto en el puntaje, y por qué.
- Análisis cualitativo del agente final apoyado en el video: ¿aprendió a esquivar proyectiles?, ¿prioriza ciertos invasores?, ¿se queda atascado en una esquina?
- Limitaciones del enfoque y del cómputo/tiempo disponible para entrenar (Atari suele requerir millones de pasos para converger a un buen desempeño).
- Reflexión sobre el trade-off exploración/explotación observado (p. ej. comportamiento errático mientras $\epsilon$ es alto vs. estancamiento si decae demasiado rápido).

## 7. Conclusiones

**TODO**:

- Desempeño final del agente (puntaje promedio de evaluación) e interpretación en el contexto del juego.
- Principales aprendizajes técnicos y metodológicos.
- Posibles mejoras futuras (más pasos de entrenamiento, PPO/A2C, prioritized experience replay, más ajuste de hiperparámetros, etc.).

## 8. Repositorio de GitHub

https://github.com/AngelEsquit/PRY2-DL

Ver `README.md` del repositorio para instrucciones detalladas de instalación, entrenamiento y evaluación/carga del modelo final.